In [2]:
!pip install -q --no-cache-dir unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 151.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 294.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 173.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 168.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 167.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 294.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 188.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 247.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 262.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install bitsandbytes torch torchvision torchaudio transformers accelerate huggingface_hub datasets "trl==0.19.1" peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 12.3 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


In [4]:
import torch


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-4B",
    max_seq_length=384,
    load_in_4bit=True,
)

model.config.use_cache = False
print("✅ Model loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✅ Model loaded successfully


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=384,
)

print("✅ LoRA adapters attached")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
✅ LoRA adapters attached
Trainable parameters: 21,233,664


In [8]:
from datasets import load_dataset


dataset = load_dataset(
    "json",
    data_files={"train": "finetuning_data_sft_prompt_completion.jsonl"},
    split="train"
)

print(f"✅ Dataset loaded: {len(dataset)} records")
print("\n--- Sample record ---")
print(f"Prompt: {dataset[0]['prompt']}")
print(f"Completion: {dataset[0]['completion']}")


Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded: 317 records

--- Sample record ---
Prompt: System:
You are a helpful customer support assistant for NUST Bank. Answer the customer's question accurately and concisely using the bank's product knowledge.

User:
I would like to open an account with my son, do u have any product for kids?
Completion: Yes our product is Little Champs Account. It is designed specifically for minors (individuals below the age of 18 years). A child requires the help of a parental/legal guardian to open this account and avail its facilities. Little Champs get a Debit Card and chequebook which is free the first time


In [9]:
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 317
})

In [10]:
def format_example(example):
    text = (
        f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['completion']}<|im_end|>{tokenizer.eos_token}"
    )
    return {"text": text}

# Keep only a single text column so SFTTrainer treats this as plain LM text data.
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print("✅ Dataset formatted")
print("\n--- Formatted sample ---")
print(dataset[0]["text"])
print("\nColumns:", dataset.column_names)

Map:   0%|          | 0/317 [00:00<?, ? examples/s]

✅ Dataset formatted

--- Formatted sample ---
<|im_start|>user
System:
You are a helpful customer support assistant for NUST Bank. Answer the customer's question accurately and concisely using the bank's product knowledge.

User:
I would like to open an account with my son, do u have any product for kids?<|im_end|>
<|im_start|>assistant
Yes our product is Little Champs Account. It is designed specifically for minors (individuals below the age of 18 years). A child requires the help of a parental/legal guardian to open this account and avail its facilities. Little Champs get a Debit Card and chequebook which is free the first time<|im_end|><|im_end|>

Columns: ['text']


In [11]:
dataset.column_names

['text']

In [ ]:
from trl import SFTTrainer, SFTConfig
import gc
import torch

# Clear stale objects before a fresh training run.
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=384,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        warmup_steps=10,
        num_train_epochs=3,
        packing=False,
        logging_steps=5,
        save_steps=1000,
        save_strategy="no",
        output_dir="/home/Ubuntu/finetune/outputs",
        optim="adamw_torch",
        seed=3407,
        dataset_num_proc=1,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer_stats = trainer.train()
print("✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Sample packing skipped (processor-based model detected).
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/317 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 317 | Num Epochs = 3 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
5,3.075185
10,2.736627
15,2.390743
20,1.876337
25,1.659810
30,1.578495


OutOfMemoryError: CUDA out of memory. Tried to allocate 46.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 25.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.34 GiB is allocated by PyTorch, and 32.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from trl import SFTTrainer, SFTConfig
from IPython.display import display, Markdown

display(Markdown("## 🚀 Starting Fine-tuning (T4-safe profile)"))

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=512,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        packing=False,
        logging_steps=5,
        output_dir="/home/Ubuntu/finetune/outputs",
        optim="adamw_torch",
        seed=3407,
        dataset_num_proc=1,
        save_strategy="no",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer_stats = trainer.train()
print("✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")


In [ ]:
from google.colab import drive
import os
from IPython.display import display, Markdown

display(Markdown("## 💾 Saving Fine-tuned Model to Kaggle Output"))

# Use Kaggle's working directory (persists after notebook execution)
output_base = "/kaggle/working"
os.makedirs(output_base, exist_ok=True)

# Create model subdirectories
lora_path = f"{output_base}/qwen35_lora"
merged_path = f"{output_base}/qwen35_merged"

os.makedirs(lora_path, exist_ok=True)
os.makedirs(merged_path, exist_ok=True)

print(f"Saving to: {output_base}")

# Save LoRA adapters
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to {lora_path}")

# Save merged 16-bit model for inference
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit"
)
print(f"✅ Merged 16-bit model saved to {merged_path}")

print(f"\n✅ All models saved to Kaggle output!")
print(f"   LoRA: {lora_path}")
print(f"   Merged: {merged_path}")
print(f"\n💡 Download these folders from the Output tab after notebook completion.")
